In [1]:
# ============================================================
# SFT (QLoRA) for Qwen/Qwen3-8B: Roleplayer JSON answers
#   - Saves LoRA adapter to ./qwen3_roleplayer_sft_lora
# ============================================================

import os, json, random, re, inspect, sys, subprocess
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

os.environ["CUDA_VISIBLE_DEVICES"] = "3"   # adjust if needed

# ---------- Install/upgrade deps if needed ----------
def _pip_install(pkgs: List[str]):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

def _version_tuple(v: str):
    nums = []
    for x in re.findall(r"\d+", v.split("+")[0]):
        nums.append(int(x))
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums[:3])

try:
    import torch
    import transformers
except Exception:
    _pip_install(["torch", "transformers"])
    import torch, transformers

import transformers as _tf
if _version_tuple(_tf.__version__) < (4, 45, 0):
    _pip_install(["-U", "transformers"])
    import importlib
    importlib.reload(_tf)

try:
    import datasets
except Exception:
    _pip_install(["datasets"])
try:
    import peft
except Exception:
    _pip_install(["peft"])
try:
    import bitsandbytes
except Exception:
    _pip_install(["bitsandbytes"])

from datasets import Dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ---------- Repro ----------
SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

# ---------- Paths ----------
BENCH_CANDS = [
    "./11fix.json", "./22fix.json", "./44fix.json",
    "/mnt/data/11fix.json", "/mnt/data/22fix.json",
    "/mnt/data/44fix.json", "/mnt/data/44fix (1).json",
]
OUT_DIR = "./qwen3_roleplayer_sft_lora"
MODEL_NAME = "Qwen/Qwen3-8B"

TRAIN_SPLIT_PATH = "./bench_train_80.json"
TEST_SPLIT_PATH  = "./bench_test_20.json"

# ---------- Slots / question templates ----------
ALL_SLOTS = ["Year","MSRP","city mpg","highway MPG","Engine Fuel Type","Vehicle Size","Model"]

QUESTION_TEMPLATES = {
    "Year": "What is your minimum model year? Give a year number, or say you have no preference.",
    "MSRP": "What is your maximum budget (MSRP)? Give a number, or say you have no preference.",
    "city mpg": "What minimum city MPG do you want? Give a number, or say you have no preference.",
    "highway MPG": "What minimum highway MPG do you want? Give a number, or say you have no preference.",
    "Engine Fuel Type": "Do you have a preferred engine fuel type? If yes, state it; otherwise say no preference.",
    "Vehicle Size": "Do you have a preferred vehicle size (e.g., Compact, Midsize, Large)? If yes, state it; otherwise say no preference.",
    "Model": "Do you have a specific model in mind? If yes, state it; otherwise say no preference.",
}

# ---------- Importance label mapping ----------
def weight_to_label(w: float) -> str:
    if w >= 0.85: return "MUST_HAVE"
    if w >= 0.65: return "STRONG_PREFERENCE"
    if w >= 0.40: return "NICE_TO_HAVE"
    if w >= 0.15: return "FLEXIBLE"
    return "FLEXIBLE"

# ---------- Load benchmark(s) (mix + dedup) ----------
def load_all_benchmarks() -> List[Dict[str, Any]]:
    paths = [p for p in BENCH_CANDS if os.path.exists(p)]
    if not paths:
        raise FileNotFoundError(f"Could not find any benchmark files. Looked in: {BENCH_CANDS}")

    all_items = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError(f"{p} is not a JSON list.")
        all_items.extend(data)

    # de-dup by (base_query_sentence, persona)
    uniq = {}
    for it in all_items:
        k = (it.get("base_query_sentence",""), it.get("persona",""))
        uniq[k] = it
    merged = list(uniq.values())
    print(f"Loaded {len(all_items)} total items -> {len(merged)} unique items from {len(paths)} files:\n  " + "\n  ".join(paths))
    return merged

bench_all = load_all_benchmarks()

# ---------- Item-level 80/20 split (seeded) ----------
rng = random.Random(SEED)
idxs = list(range(len(bench_all)))
rng.shuffle(idxs)
cut = int(0.8 * len(idxs))
train_idxs = set(idxs[:cut])
test_idxs  = set(idxs[cut:])

bench_train = [bench_all[i] for i in range(len(bench_all)) if i in train_idxs]
bench_test  = [bench_all[i] for i in range(len(bench_all)) if i in test_idxs]

with open(TRAIN_SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(bench_train, f, ensure_ascii=False, indent=2)
with open(TEST_SPLIT_PATH, "w", encoding="utf-8") as f:
    json.dump(bench_test, f, ensure_ascii=False, indent=2)

print(f"\nSaved item-level splits:")
print(f"  Train items: {len(bench_train)} -> {TRAIN_SPLIT_PATH}")
print(f"  Test  items: {len(bench_test)} -> {TEST_SPLIT_PATH}")

# ---------- Build per-item maps from ground truth ----------
def build_truth_maps(item: Dict[str, Any]) -> Tuple[Dict[str, Dict[str,Any]], Dict[Tuple[str,str,str], float]]:
    slot_to_constraint = {}
    key_to_weight = {}

    add = item.get("additional_constraints", []) or []
    for c in add:
        if isinstance(c, dict) and c.get("column") in ALL_SLOTS:
            slot_to_constraint[c["column"]] = {"column": c["column"], "op": c["op"], "value": c["value"]}

    cw = item.get("constraint_weights", []) or []
    for c in cw:
        if not isinstance(c, dict):
            continue
        col, op, val = c.get("column"), c.get("op"), c.get("value")
        w = c.get("weight", None)
        if col in ALL_SLOTS and op in ("==",">=","<=") and w is not None:
            try:
                key_to_weight[(col, op, str(val))] = float(w)
            except Exception:
                pass

    return slot_to_constraint, key_to_weight

# ---------- Roleplayer system prompt (SFT target behavior) ----------
ROLEPLAYER_SFT_SYS = (
    "You are ROLEPLAYER_USER.\n"
    "You ONLY know the provided persona and the base query.\n"
    "You must answer the assistant's question using ONLY information stated or clearly implied in the persona.\n"
    "CRITICAL:\n"
    " - If the persona includes a concrete value relevant to the asked slot, you MUST state that value explicitly.\n"
    " - Only if the persona truly contains no information about that slot, say you have no preference.\n"
    "Output MUST be valid JSON ONLY, matching the given schema.\n"
    "Do NOT include any extra keys.\n"
)

def make_user_prompt(base_query: str, persona: str, slot: str) -> str:
    q = QUESTION_TEMPLATES.get(slot, f"What is your preference for {slot}?")
    return (
        f"Base query:\n{base_query}\n\n"
        f"Persona:\n{persona}\n\n"
        f"Assistant question (slot={slot}):\n{q}\n\n"
        "Return ONLY JSON in this schema:\n"
        '{\n'
        '  "slot": "<slot>",\n'
        '  "has_preference": true/false,\n'
        '  "op": "=="|">="|"<=",\n'
        '  "value": string|int|null,\n'
        '  "importance_label": "MUST_HAVE"|"STRONG_PREFERENCE"|"NICE_TO_HAVE"|"FLEXIBLE"|"NO_PREFERENCE"\n'
        '}\n'
    )

def make_target_json(slot: str,
                     slot_to_constraint: Dict[str, Dict[str,Any]],
                     key_to_weight: Dict[Tuple[str,str,str], float]) -> str:
    if slot not in slot_to_constraint:
        obj = {
            "slot": slot,
            "has_preference": False,
            "op": "==",
            "value": None,
            "importance_label": "NO_PREFERENCE"
        }
        return json.dumps(obj, ensure_ascii=False)

    c = slot_to_constraint[slot]
    key = (c["column"], c["op"], str(c["value"]))
    w = float(key_to_weight.get(key, 0.5))
    label = weight_to_label(w)

    obj = {
        "slot": slot,
        "has_preference": True,
        "op": c["op"],
        "value": c["value"],
        "importance_label": label
    }
    return json.dumps(obj, ensure_ascii=False)

# ---------- Build SFT examples ----------
def build_examples(bench: List[Dict[str,Any]], negatives_per_item: int = 2) -> List[Dict[str,Any]]:
    ex = []
    rng_local = random.Random(SEED)

    for it in bench:
        base_query = it.get("base_query_sentence", "").strip()
        persona = (it.get("persona", "") or "").strip()
        if not base_query or not persona:
            continue

        slot_to_constraint, key_to_weight = build_truth_maps(it)
        pos_slots = list(slot_to_constraint.keys())

        # Positives: for every slot actually present (2 for 22fix, 4 for others)
        for slot in pos_slots:
            ex.append({
                "system": ROLEPLAYER_SFT_SYS,
                "user": make_user_prompt(base_query, persona, slot),
                "assistant": make_target_json(slot, slot_to_constraint, key_to_weight),
                "slot": slot,
                "is_positive": 1
            })

        # Negatives: teach NO_PREFERENCE schema on a couple missing slots
        neg_cands = [s for s in ALL_SLOTS if s not in slot_to_constraint]
        rng_local.shuffle(neg_cands)
        for slot in neg_cands[:max(0, negatives_per_item)]:
            ex.append({
                "system": ROLEPLAYER_SFT_SYS,
                "user": make_user_prompt(base_query, persona, slot),
                "assistant": make_target_json(slot, slot_to_constraint, key_to_weight),
                "slot": slot,
                "is_positive": 0
            })

    rng_local.shuffle(ex)
    print(f"Built {len(ex)} SFT examples from {len(bench)} items "
          f"(positives={sum(e['is_positive'] for e in ex)}, negatives={sum(1-e['is_positive'] for e in ex)})")
    return ex

train_examples = build_examples(bench_train, negatives_per_item=2)
test_examples  = build_examples(bench_test,  negatives_per_item=2)

print(f"\nExample counts:")
print(f"  Train examples: {len(train_examples)}")
print(f"  Test  examples: {len(test_examples)}")

train_ds = Dataset.from_list(train_examples)
test_ds  = Dataset.from_list(test_examples)

# ---------- Tokenizer + model (4-bit QLoRA) ----------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
mdl.config.use_cache = False
mdl = prepare_model_for_kbit_training(mdl, use_gradient_checkpointing=True)

# ---------- LoRA target module discovery (robust) ----------
def guess_lora_targets(model) -> List[str]:
    preferred = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    found = set()
    for name, module in model.named_modules():
        if module.__class__.__name__ == "Linear":
            last = name.split(".")[-1]
            if last in preferred:
                found.add(last)
    return sorted(found) if found else preferred

target_modules = guess_lora_targets(mdl)
print("LoRA target_modules:", target_modules)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

mdl = get_peft_model(mdl, lora_cfg)
mdl.print_trainable_parameters()

# ---------- Tokenization with completion-only loss masking ----------
MAX_LEN = 1024

def tokenize_row(row: Dict[str,Any]) -> Dict[str,Any]:
    system = row["system"]
    user = row["user"]
    assistant = row["assistant"]

    prompt_msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt_text = tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    full_msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]
    full_text = tok.apply_chat_template(
        full_msgs,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    prompt_ids = tok(prompt_text, add_special_tokens=False).input_ids
    full = tok(full_text, add_special_tokens=False)

    input_ids = full.input_ids[:MAX_LEN]
    attn = full.attention_mask[:MAX_LEN]

    prompt_len = min(len(prompt_ids), len(input_ids))
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    labels = labels[:MAX_LEN]

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

train_tok = train_ds.map(tokenize_row, remove_columns=train_ds.column_names)
test_tok  = test_ds.map(tokenize_row,  remove_columns=test_ds.column_names)

# ---------- Collator (pads labels with -100) ----------
def collate(features: List[Dict[str,Any]]) -> Dict[str,torch.Tensor]:
    maxlen = max(len(f["input_ids"]) for f in features)
    input_ids, attn, labels = [], [], []
    for f in features:
        pad = maxlen - len(f["input_ids"])
        input_ids.append(f["input_ids"] + [tok.pad_token_id]*pad)
        attn.append(f["attention_mask"] + [0]*pad)
        labels.append(f["labels"] + [-100]*pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

# ---------- TrainingArguments: version-robust kwargs ----------
sig = inspect.signature(TrainingArguments.__init__)
has_eval_strategy = "eval_strategy" in sig.parameters
has_evaluation_strategy = "evaluation_strategy" in sig.parameters
has_report_to = "report_to" in sig.parameters
has_optim = "optim" in sig.parameters

ta_kwargs = dict(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_steps=200,
    eval_steps=200,
    save_total_limit=2,
    fp16=True,
)

if has_eval_strategy:
    ta_kwargs["eval_strategy"] = "steps"
elif has_evaluation_strategy:
    ta_kwargs["evaluation_strategy"] = "steps"

ta_kwargs["save_strategy"] = "steps"

if has_report_to:
    ta_kwargs["report_to"] = "none"

if has_optim:
    ta_kwargs["optim"] = "paged_adamw_32bit"

if "gradient_checkpointing" in sig.parameters:
    ta_kwargs["gradient_checkpointing"] = True

args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=mdl,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,   # IMPORTANT: eval on HELD-OUT TEST split
    data_collator=collate,
)

# ---------- Train ----------
trainer.train()

# ---------- Save adapter ----------
os.makedirs(OUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print(f"\nSaved LoRA adapter + tokenizer to: {OUT_DIR}")

# ---------- Quick sanity check on a few TEST examples ----------
def try_parse_json(s: str) -> bool:
    s = s.strip()
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return False
    try:
        json.loads(s[start:end+1])
        return True
    except Exception:
        return False

mdl.eval()
sample_n = min(8, len(test_examples))
print("\n--- SANITY CHECK (TEST samples) ---")
for i in range(sample_n):
    ex = test_examples[i]
    prompt_msgs = [
        {"role":"system","content":ex["system"]},
        {"role":"user","content":ex["user"]},
    ]
    prompt_text = tok.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt_text, return_tensors="pt").to(mdl.device)

    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )
    gen = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    print(f"\nSample {i+1}/{sample_n} | slot={ex['slot']} | expected_positive={ex['is_positive']}")
    print("Generated:", gen)
    print("JSON parses:", try_parse_json(gen))


Loaded 122 total items -> 122 unique items from 3 files:
  ./11fix.json
  ./22fix.json
  ./44fix.json

Saved item-level splits:
  Train items: 97 -> ./bench_train_80.json
  Test  items: 25 -> ./bench_test_20.json
Built 518 SFT examples from 97 items (positives=324, negatives=194)
Built 132 SFT examples from 25 items (positives=82, negatives=50)

Example counts:
  Train examples: 518
  Test  examples: 132


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

LoRA target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Saved LoRA adapter + tokenizer to: ./qwen3_roleplayer_sft_lora

--- SANITY CHECK (TEST samples) ---

Sample 1/8 | slot=Vehicle Size | expected_positive=1
Generated: {"slot": "Vehicle Size", "has_preference": true, "op": "==", "value": "Midsize", "importance_label": "FLEXIBLE"}
JSON parses: True

Sample 2/8 | slot=highway MPG | expected_positive=1
Generated: {"slot": "highway MPG", "has_preference": true, "op": ">=", "value": 34, "importance_label": "FLEXIBLE"}
JSON parses: True

Sample 3/8 | slot=city mpg | expected_positive=0
Generated: {"slot": "city mpg", "has_preference": false, "op": "==", "value": null, "importance_label": "NO_PREFERENCE"}
JSON parses: True

Sample 4/8 | slot=Vehicle Size | expected_positive=1
Generated: {"slot": "Vehicle Size", "has_preference": true, "op": "==", "value": "Midsize", "importance_label": "FLEXIBLE"}
JSON parses: True

Sample 5/8 | slot=Model | expected_positive=1
Generated: {"slot": "Model", "has_preference": true, "op": "==", "value": "CLS-Class